In [1]:
from gurobipy import *
import pandas as pd 
import numpy as np 

mod = Model('Bus Optimization')

club_filename = "/voc/data/club_data.xlsx"
routing_filename = "/voc/data/routing_solutions.csv"

Restricted license - for non-production use only - expires 2026-11-23


In the following cell, implement the integer program written in Section 1, using the "mod" object defined above.

In [2]:
# read data
capacities = [14, 15, 24, 30, 66] # k
club_df = pd.read_excel(club_filename, header=1)
route_df = pd.read_csv(routing_filename, header=1)
club_df.columns = [str(i) for i in club_df.columns]
route_df.columns = [str(i) for i in route_df.columns]
club_names = club_df["Club"].tolist() # c
clubs = {}
for _, row in club_df.iterrows():
    club = row["Club"]
    clubs[club] = {
        "students": int(row["Students"]), # s_c
        "orig": {i: int(row[str(i)]) for i in capacities} # b_c,j for section 2
    }
total_students = sum(clubs[i]["students"] for i in club_names)
fleet = {j: sum(clubs[i]["orig"][j] for i in club_names) for j in capacities} # b_j
routes = {}
for _, row in route_df.iterrows():
    r = int(row["ID"])
    club = row["Club"]
    routes[r] = {
        "club": club, # c(r)
        "req": {k: int(row[str(k)]) for k in capacities}, # a_r,k
        "time": float(row["Avg Srvc Time"]) # t_r
    }
route_ids = list(routes.keys()) # r
routes_by_club = {c:[] for c in club_names} # r_c
for r in route_ids:
    routes_by_club[routes[r]["club"]].append(r)

# Decision variables
x = mod.addVars(route_ids, vtype=GRB.BINARY, name="x")
y = mod.addVars(club_names, capacities, vtype=GRB.INTEGER, name="y")

# Objective. Minimize average individual service time
mod.setObjective(quicksum(clubs[routes[r]["club"]]["students"] * routes[r]["time"] * x[r] for r in route_ids) / total_students, GRB.MINIMIZE)

# Constraints
# Each club must have exactly one routing solution
for c in club_names:
    mod.addConstr(quicksum(x[r] for r in routes_by_club[c]) == 1)
    
# Each club must have >= number of buses of the capacity required by the routing solution
for c in club_names:
    for k in capacities:
        mod.addConstr(quicksum(y[c,j] for j in capacities if j >= k) >= quicksum(sum(routes[r]["req"][h] for h in capacities if h >= k) * x[r]
        for r in routes_by_club[c]))

# All buses of each capacity are assigned
for j in capacities:
    mod.addConstr(quicksum(y[c,j] for c in club_names) == fleet[j])

# Section 2 constraint
for c in club_names:
    for j in capacities:
        mod.addConstr(y[c,j] == clubs[c]["orig"][j]) 

mod.optimize()

if mod.status == GRB.OPTIMAL:
    print("Optimal average service time:", mod.objVal)

# Section 3
if mod.status == GRB.OPTIMAL:
    print("Optimal average service time:", mod.objVal)
    print("\nBus assignments by club:")
    for c in club_names:
        vals = {j: int(round(y[c,j].X)) for j in capacities}
        print(c, vals)

    print("\nSelected routes:")
    for r in route_ids:
        if x[r].X > 0.5:
            print(r, routes[r]["club"], routes[r]["req"], routes[r]["time"])

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (linux64 - "Ubuntu 20.04.6 LTS")

CPU model: Intel(R) Xeon(R) Platinum 8375C CPU @ 2.90GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 192 rows, 163 columns and 730 nonzeros
Model fingerprint: 0xfe431768
Variable types: 0 continuous, 163 integer (78 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+00]
  Objective range  [9e-02, 3e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+01]
Found heuristic solution: objective 22.3227827
Presolve removed 192 rows and 163 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 2: 18.4295 22.3228 

Optimal solution found (tolerance 1.00e-04)
Best objective 1.842948197748e+01, best bound 1.842948197748e+01, gap 0.0000%
Optimal 

In the following cell, save your optimal objective value to the opt_obj_val object.

In [3]:
opt_obj_val = mod.objVal